# Derive underwriting features

Read persisted source tables, validate, and recompute feature rows. This small demo collects source rows locally; replace with distributed aggregation for production volumes.


In [ ]:
import sys
from pathlib import Path
BASE = Path('/lakehouse/default/Files/underwriting_demo')
sys.path.insert(0, str(BASE / 'src'))
from insurance_demo import build_data, validate, column_type, AS_OF, ROOT
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType
TABLE_PREFIX = 'demo_'

def save_rows(name, rows):
    types = {'bit': BooleanType(), 'bigint': LongType(), 'float': DoubleType()}
    schema = StructType([StructField(field, types.get(column_type(field, rows), StringType()), True) for field in rows[0]])
    frame = spark.createDataFrame(rows, schema)
    for field in rows[0]:
        datatype = column_type(field, rows)
        if datatype == 'date':
            frame = frame.withColumn(field, F.to_date(field))
        elif datatype == 'datetime2(6)':
            frame = frame.withColumn(field, F.to_timestamp(field))
    destination = TABLE_PREFIX + name
    if spark.catalog.tableExists(destination):
        properties = {row.key: row.value for row in spark.sql(f'SHOW TBLPROPERTIES {destination}').collect()}
        assert properties.get('delta.columnMapping.mode', 'none') == 'none', 'Use a new managed demo table without column mapping'
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(destination)
    assert spark.table(destination).count() == len(rows)
    return frame

from insurance_demo import calculate_features
data = {name: [row.asDict() for row in spark.table(TABLE_PREFIX + name).collect()] for name in build_data()}
for rows in data.values():
    for row in rows:
        for key, value in row.items():
            if hasattr(value, 'isoformat'):
                row[key] = value.isoformat()
validate(data)
features = calculate_features(data)
save_rows('UnderwritingFeatures', features)
display(spark.table(TABLE_PREFIX + 'UnderwritingFeatures'))
